# Week 16: Advanced MLOps & Distributed Deep Learning
## Notebook 04: Knowledge Distillation & Model Pruning from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Deploying massive models into edge devices or real-time production inference (<20ms latency) requires model compression without sacrificing predictive capability.

Two foundational techniques:
1. **Knowledge Distillation (Hinton et al.):** Transfers dark knowledge from a large teacher model to a compact student model using softened probability distributions.
2. **Magnitude Pruning:** Removes redundant weights with near-zero values, inducing sparsity.

---

### 2. Mathematical Foundations: Knowledge Distillation
Given teacher logits $z_t$ and student logits $z_s$:

1. **Softened Probabilities with Temperature $T$:**
   $$p_i(T) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
   - When $T=1$, standard sharp softmax.
   - When $T > 1$, probability distribution softens, exposing secondary relationships (e.g. "this image is 80% BMW, but 15% Audi and 0.0001% Tree"). This is the **Dark Knowledge**!

2. **Combined Distillation Loss:**
   $$\mathcal{L}_{total} = \alpha T^2 \mathcal{L}_{KL}\left(p_s(T) \parallel p_t(T)\right) + (1 - \alpha) \mathcal{L}_{CE}(y, p_s(1))$$
   The $T^2$ multiplier scales gradients back to match standard cross-entropy magnitude.

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate Teacher Model and Student Model Logits
# 3 classes: Cat (0), Dog (1), Car (2)
batch_size = 4
num_classes = 3

# Teacher has high confidence but provides soft cross-class correlations
teacher_logits = np.array([
    [8.0, 4.5, -2.0],  # Cat with subtle dog features
    [4.0, 7.5, -1.0],  # Dog with subtle cat features
    [-3.0, -2.0, 9.0], # Distinct Car
    [7.0, 5.0, -3.0]   # Cat
])

# Compact student initial random logits
student_logits = np.random.randn(batch_size, num_classes)
true_labels = np.array([0, 1, 2, 0])

def softmax_temp(logits, T):
    exp_l = np.exp((logits - np.max(logits, axis=1, keepdims=True)) / T)
    return exp_l / np.sum(exp_l, axis=1, keepdims=True)

# Step 2: Compare Sharp vs Softened Distributions (T=1 vs T=4)
print("Teacher Sharp Probabilities (T=1):")
print(np.round(softmax_temp(teacher_logits, T=1.0), 3))

print("\nTeacher Softened Probabilities (Dark Knowledge exposed at T=4):")
soft_teacher = softmax_temp(teacher_logits, T=4.0)
print(np.round(soft_teacher, 3))

# Step 3: Compute Distillation Loss
T = 4.0
alpha = 0.7  # 70% teacher guidance, 30% ground truth

soft_student = softmax_temp(student_logits, T=T)
sharp_student = softmax_temp(student_logits, T=1.0)

# KL Divergence: sum(P_teacher * log(P_teacher / P_student))
kl_div = np.sum(soft_teacher * np.log((soft_teacher + 1e-9) / (soft_student + 1e-9)), axis=1).mean()
distill_loss = (T ** 2) * kl_div

# Hard Cross-Entropy Loss
ce_loss = -np.mean(np.log(sharp_student[np.arange(batch_size), true_labels] + 1e-9))

total_loss = alpha * distill_loss + (1 - alpha) * ce_loss
print(f"\nDistillation KL Loss: {distill_loss:.4f}")
print(f"Hard Cross-Entropy Loss: {ce_loss:.4f}")
print(f"Total Student Loss: {total_loss:.4f}")

# Step 4: Weight Magnitude Pruning (Unstructured 50% Sparsity)
W = np.random.randn(6, 6)
print("\nOriginal Weight Matrix (6x6):")
print(np.round(W[:3, :3], 3))

sparsity_target = 0.50
threshold = np.percentile(np.abs(W), sparsity_target * 100)
mask = np.abs(W) >= threshold
W_pruned = W * mask

actual_sparsity = np.mean(W_pruned == 0.0) * 100
print(f"Pruned Matrix (Threshold: {threshold:.3f}):")
print(np.round(W_pruned[:3, :3], 3))
print(f"Achieved Sparsity: {actual_sparsity:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Temperature Softening:** At $T=4$, probabilities for similar classes (Cat vs Dog) soften, transferring semantic similarity information to the student.
2. **$T^2$ Scaling:** Softening dividing logits by $T$ reduces gradient magnitude by $1/T^2$; multiplying by $T^2$ restores balance.
3. **Weight Pruning:** `np.percentile` finds the magnitude cutoff, setting the lowest 50% of weights to 0.0.

---

### 4. Industry Applications & Trade-offs
- **DistilBERT:** 40% smaller and 60% faster than BERT while retaining 97% of language understanding capabilities.
- **Structured vs Unstructured Pruning:** Unstructured pruning leaves zeros in arbitrary locations requiring specialized sparse BLAS libraries. Structured pruning drops entire channels/heads, achieving immediate speedups on standard GPUs.

---

### 5. Top Interview Questions & Answers
**Q1: Why do we multiply the KL loss by $T^2$ during distillation?**
> **Answer:** Dividing logits by $T$ scales gradients by $1/T$. In backpropagation, both probabilities and logits scale, resulting in gradients scaled by $1/T^2$. Multiplying by $T^2$ ensures teacher gradients match hard label cross-entropy gradients.

**Q2: What is the Lottery Ticket Hypothesis (Frankle & Carbin)?**
> **Answer:** A randomly-initialized dense neural network contains a sub-network (a "winning ticket") that, when trained in isolation from initialization, can match the accuracy of the original dense network in comparable training iterations.